# L10 · NB 01 — Monday morning: feel the power of pretrained transformers

> *Marcus's Friday question is on Sarah's mind: build a shopping assistant. Before she designs anything, she opens her laptop and explores what pretrained transformer models can already do — out of the box, in one line each.*

This pre-class notebook is your hands-on introduction to the modern NLP stack. We will use three pretrained pipelines:

1. **Sentiment analysis** — is a review positive or negative?
2. **Named-entity recognition (NER)** — extract people, places, organisations
3. **Zero-shot classification** — sort text into labels you describe, with no training

Each is one line of Python. Each runs a 67-400M parameter transformer underneath. Welcome to the era of pretrained models.

## 1 · Setup

In [ ]:
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # prevent tokenizer deadlock on macOS + VS Code

import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import pipeline

torch.set_num_threads(1)
print('Torch:', torch.__version__)

## 2 · Sentiment analysis

The classic NLP starter. Is this product review positive or negative?

In [ ]:
sentiment = pipeline('sentiment-analysis')

reviews = [
    "I love this dress! Perfect for a beach holiday and the fabric is so soft.",
    "Returned. The colour in the photo was MUCH brighter than what arrived.",
    "It's fine. Does the job. Nothing special.",
    "Worst purchase I've made this year. Stitching unravelled within two wears.",
    "Got it as a gift and she loves it — thank you so much!",
]

for r in reviews:
    result = sentiment(r)[0]
    print(f"  [{result['label']:<8s} {result['score']:.3f}]  {r[:60]}")

**One line of Python** — and notice the result is mostly right, but not perfectly so. Reread review 2: *"Returned. The colour in the photo was MUCH brighter than what arrived."* That's a complaint. The model labelled it POSITIVE with 0.998 confidence. The word *brighter* and the absence of explicitly negative words tricked it.

This is a real failure mode of off-the-shelf models. They're shockingly capable on the easy 90%, and they hallucinate confidence on the 10%. Production use means knowing where the gaps are.

What's running underneath? `pipeline('sentiment-analysis')` defaults to `distilbert-base-uncased-finetuned-sst-2-english` — a 67M-parameter transformer fine-tuned on the Stanford Sentiment Treebank (movie reviews). Loads ~268 MB the first time. Because it was trained on movie reviews, it does fine on product reviews that LOOK like movie reviews but struggles on retail-specific patterns like "I returned this". A small fine-tune on YOUR review corpus would fix this.

That's the modern NLP stack: someone has trained a model for you; you import and use it; and you ALWAYS run a calibration pass on your own data.

## 3 · Named-entity recognition (NER)

Extract people, places, organisations, dates from text. Useful for tagging news articles, customer support tickets, product descriptions.

In [ ]:
ner = pipeline('ner', model='dslim/bert-base-NER', aggregation_strategy='simple')

text = "Sarah Chen from NorthStar Retail launched a new search feature in London on Monday. The team thanked Marcus Patel for sponsoring the project."

results = ner(text)
print(f"Input: {text}\n")
print(f"{'Entity':<25s} {'Type':<10s} {'Score':>8s}")
print('-' * 50)
for r in results:
    print(f"  {r['word']:<25s} {r['entity_group']:<10s} {r['score']:>8.3f}")

The model isn't perfect (it might miss a name, mis-classify a city), but it's getting most of these right with **zero training on your data**. NER is a textbook task; pretrained models nail it.

Production use cases: redacting PII from logs, tagging support tickets for routing, extracting people/companies from news for a feed.

## 4 · Zero-shot classification

The third demo is more impressive than it sounds. **Zero-shot** means you classify text into categories the model has never been explicitly trained on — you just *describe* the labels and the model decides which one fits.

In [ ]:
zsc = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

text = "Returned the shirt after one wash — the colour ran and now my whites are pink."
candidate_labels = ["quality complaint", "praise", "shipping issue", "size issue", "billing question"]

result = zsc(text, candidate_labels)
print(f"Input: {text}\n")
print(f"{'Label':<25s} {'Score':>8s}")
print('-' * 36)
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:<25s} {score:>8.3f}")

Same model, completely different label set — just by changing the candidate strings. This is shockingly useful in production: you can build a customer-support ticket router in 30 minutes without labelled training data.

Underneath, the model (`bart-large-mnli`) was trained on natural-language-inference data — predicting whether a hypothesis is *entailed*, *contradicted*, or *neutral* given a premise. Zero-shot classification reformulates "is this text about X?" as "does this text entail the statement 'This is about X'?". Clever.

**The pattern emerging:** every one of these used to require custom training. Today: pip install, import, call. The models are pretrained, instruction-tuned, fine-tuned, and waiting on Hugging Face.

## 5 · The pattern: hub + pipeline

Hugging Face hosts **hundreds of thousands of pretrained models** for every NLP task imaginable. The `pipeline()` function from `transformers` is the simplest API:

```python
from transformers import pipeline
model = pipeline("task-name")           # uses default model for the task
model = pipeline("task-name", model=    # specify any model from the Hub
                 "user/model-id")
result = model(input_text)              # run it
```

Available tasks include:

- `sentiment-analysis`, `text-classification`
- `ner`, `token-classification`
- `zero-shot-classification` (label without training data)
- `fill-mask` (BERT-style "Paris is the [MASK] of France")
- `feature-extraction` (raw embeddings)
- `text-generation` (LLMs — we'll meet these in NB 03)
- `image-classification`, `image-segmentation`, `object-detection`
- `automatic-speech-recognition`, `text-to-speech`

You can build a surprising amount of product on these pipelines alone. No model training required.

## 6 · Three thought-questions for class

1. The sentiment pipeline labelled reviews 1, 2, 3, 4, 5 with high confidence. **Predict** what it would return for: *"It's fine, I guess. Wouldn't buy again."* — then run it and see if you were right.
2. The NER model knew "Sarah Chen" was a PER (person), "NorthStar Retail" was an ORG, "London" was a LOC. **How** do you think it learnt those distinctions? What did its training data look like?
3. If you wanted to build a product-review sentiment classifier specific to YOUR domain (e.g., recognising that "the elastic was loose" is negative), would you (a) prompt a general LLM, (b) use this off-the-shelf pipeline, (c) fine-tune a small model on your reviews? What's the trade-off?

In [5]:
# Try your own sentence here
test = "It's fine, I guess. Wouldn't buy again."
print(sentiment(test))

[{'label': 'POSITIVE', 'score': 0.9971808195114136}]


---

**Ready for class.** You've felt the power of pretrained transformers. Tomorrow we'll open the box and see WHY they work — the attention mechanism that makes all of this possible.